# Heart Disease Prediction - Model Evaluation & Comparison (Day 6)

This notebook focuses on training all three model candidates (Logistic Regression, Decision Tree, and Random Forest) side-by-side, evaluating them across standard metrics (Accuracy, Precision, Recall, and F1-Score), and comparing their classification capabilities using ROC and AUC metrics.

## Step 1: Import Libraries

**What & Why:**
- **What:** Imported data handling, visual charting, classification estimators, and performance evaluation modules.
- **Why:** These provide the necessary classes to evaluate and compare Logistic Regression, Decision Tree, and Random Forest models under a standardized environment.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    roc_auc_score
)

## Step 2: Load Dataset

**What & Why:**
- **What:** Loaded `heart_processed.csv` from the `data/` directory.
- **Why:** To fetch the cleaned and encoded tabular database ready for modeling splits.

In [ ]:
df = pd.read_csv("../data/heart_processed.csv")
df.head()

## Step 3: Prepare Data

**What & Why:**
- **What:** Separated attributes (`X`) and label (`y`), then ran an 80/20 stratified split.
- **Why:** Required structure for training inputs.

In [ ]:
X = df.drop("target", axis=1)
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## Step 4: Scale Data for Logistic Regression

**What & Why:**
- **What:** Scaled the training and test features specifically for Logistic Regression.
- **Why:** Linear models require scaled variables, whereas Tree-based models (Decision Tree and Random Forest) split nodes based on relative ordering, rendering feature scale irrelevant.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Step 5: Train All Models

**What & Why:**
- **What:** Fitted Logistic Regression, Decision Tree, and Random Forest models on training sets and generated target predictions.
- **Why:** Instantiates models under identical train-test parameters to conduct fair comparative evaluations.

In [ ]:
# Logistic Regression (trained on scaled data)
lr = LogisticRegression(random_state=42)
lr.fit(X_train_scaled, y_train)
lr_pred = lr.predict(X_test_scaled)

# Decision Tree (trained on unscaled data)
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)

# Random Forest (trained on unscaled data)
rf = RandomForestClassifier(random_state=42, n_estimators=100)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

## Step 6: Compare Metrics

**What, Why & Observations:**
- **What:** Consolidated Accuracy, Precision, Recall, and F1-Score of all models into a single DataFrame.
- **Why:** Offers a quick overview of model performances side-by-side.
- **Observations:** Random Forest achieves the highest Recall (0.87) and F1-Score (0.85). Logistic Regression achieves the highest Precision (0.83). The default Decision Tree has the lowest performance scores.

In [ ]:
results = pd.DataFrame({
    "Model":[
        "Logistic Regression",
        "Decision Tree",
        "Random Forest"
    ],
    "Accuracy":[
        accuracy_score(y_test, lr_pred),
        accuracy_score(y_test, dt_pred),
        accuracy_score(y_test, rf_pred)
    ],
    "Precision":[
        precision_score(y_test, lr_pred),
        precision_score(y_test, dt_pred),
        precision_score(y_test, rf_pred)
    ],
    "Recall":[
        recall_score(y_test, lr_pred),
        recall_score(y_test, dt_pred),
        recall_score(y_test, rf_pred)
    ],
    "F1 Score":[
        f1_score(y_test, lr_pred),
        f1_score(y_test, dt_pred),
        f1_score(y_test, rf_pred)
    ]
})
results

## Step 7: Best Model

**What & Why:**
- **What:** Sorted comparison results by accuracy to identify the top performer.
- **Why:** Highlights the best candidate model based on overall correct predictions.

In [ ]:
results.sort_values(
    by="Accuracy",
    ascending=False
)

## Step 8: ROC Curve

**What & Why:**
- **What:** Extracted the probability estimates (`predict_proba`) of the positive target class.
- **Why:** ROC curves map true positive rate vs false positive rate across all thresholds, which requires probability scores rather than binary class predictions.

In [ ]:
lr_prob = lr.predict_proba(X_test_scaled)[:,1]
dt_prob = dt.predict_proba(X_test)[:,1]
rf_prob = rf.predict_proba(X_test)[:,1]

## Step 9: Compute ROC

**What & Why:**
- **What:** Computed false positive rates (FPR) and true positive rates (TPR) for each model.
- **Why:** Provides coordinates to plot the ROC curve comparison.

In [ ]:
lr_fpr, lr_tpr, _ = roc_curve(y_test, lr_prob)
dt_fpr, dt_tpr, _ = roc_curve(y_test, dt_prob)
rf_fpr, rf_tpr, _ = roc_curve(y_test, rf_prob)

## Step 10: Plot ROC Curve

**What, Why & Observations:**
- **What:** Plotted the ROC curves for all three models and saved the chart to disk.
- **Why:** Visualizing curves lets us inspect sensitivity at different false positive rates, highlighting performance differences visually.
- **Observations:** Random Forest (blue/green curve) sits highest toward the top-left corner, suggesting it achieves the highest sensitivity with the lowest false alarm rate.

In [ ]:
plt.figure(figsize=(8,6))
plt.plot(lr_fpr, lr_tpr, label="Logistic Regression")
plt.plot(dt_fpr, dt_tpr, label="Decision Tree")
plt.plot(rf_fpr, rf_tpr, label="Random Forest")
plt.plot([0,1], [0,1], "--", color="black")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.savefig("../images/roc_curve.png", dpi=300, bbox_inches="tight")
plt.show()

## Step 11: ROC-AUC Score

**What, Why & Observations:**
- **What:** Calculated Area Under the ROC Curve (AUC) scores.
- **Why:** Quantifies model capability to distinguish between healthy and diseased classes. Higher values mean a better model.
- **Observations:** 
  - Logistic Regression AUC: **0.89** (Good)
  - Decision Tree AUC: **0.76** (Fair)
  - Random Forest AUC: **0.91** (Excellent)
  - Random Forest achieves the highest AUC, confirming superior discriminative ability.

In [ ]:
print("Logistic Regression AUC:", roc_auc_score(y_test, lr_prob))
print("Decision Tree AUC:", roc_auc_score(y_test, dt_prob))
print("Random Forest AUC:", roc_auc_score(y_test, rf_prob))

## Step 12: Visualize Model Comparison

**What & Why:**
- **What:** Plotted a bar chart comparing performance metrics across all models and saved the chart.
- **Why:** Bar charts make it easy to compare multiple metrics across models simultaneously.

In [ ]:
results.set_index("Model").plot(
    kind="bar",
    figsize=(10,6)
)
plt.title("Model Comparison")
plt.ylabel("Score")
plt.ylim(0,1)
plt.xticks(rotation=0)
plt.savefig("../images/model_comparison_chart.png", dpi=300, bbox_inches="tight")
plt.show()

## Step 13: Interpret the Results

**Observations & Interpretations:**
- **Logistic Regression:** Simple, fast, and achieves strong, robust performance metrics (Accuracy 82.07%, AUC 0.89). Highly interpretable coefficients make it a valuable clinical tool.
- **Decision Tree:** Quick to train and highly interpretable, but struggles with overfitting, leading to the lowest scores here (Accuracy 76.09%, AUC 0.76).
- **Random Forest:** Achieves the best overall classification metrics (Accuracy 82.61%, AUC 0.91) and highest recall (87.25%). This makes it the strongest candidate model, as high recall is crucial in diagnostics to minimize dangerous false negatives.

## Step 14: Save Results

**What & Why:**
- **What:** Saved the model comparison metrics DataFrame as a CSV file.
- **Why:** To save numerical results for reporting or deployment planning.

In [ ]:
# Save comparison results DataFrame to CSV
results.to_csv("../data/model_comparison.csv", index=False)

## 📝 Mini Exercise Questions & Answers

**1. Which model achieved the highest accuracy?**
- **Random Forest** achieved the highest accuracy of **82.61%**.

**2. Which model had the highest Precision?**
- **Logistic Regression** had the highest precision of **83.50%** (class 1).

**3. Which model had the highest Recall?**
- **Random Forest** had the highest recall of **87.25%** (class 1).

**4. Which model had the highest F1-score?**
- **Random Forest** had the highest F1-score of **84.76%**.

**5. Which model had the highest ROC-AUC?**
- **Random Forest** had the highest ROC-AUC of **0.91** (specifically `0.9075`), indicating excellent class separation.

**6. Which model would you recommend for heart disease prediction? Why?**
- **Random Forest** is recommended for deployment. In medical diagnostics, minimizing False Negatives (actual sick patients missed by the model) is the highest priority. Random Forest achieves the highest **Recall (87.25%)** alongside the best overall F1-Score (**84.76%**) and class separation AUC (**0.91**), making it the safest and most accurate model choice.